# ARIMA Simulation — Shredder Bearing Temperature Baseline Prediction

## 개요

이 노트북은 **ARIMA(자기회귀 누적 이동평균)** 모델을 이용한 시계열 기본 예측을 체험하는 시뮬레이션입니다.

| 항목 | 내용 |
|------|------|
| **모델** | ARIMA (AutoRegressive Integrated Moving Average) |
| **시나리오** | 슈레더 베어링 온도 90일 데이터로 미래 예측 |
| **핵심 원리** | 순수 통계 기반 — 과거 값 + 차분 + 오차 보정 |
| **역할** | 다른 AI 모델의 **Baseline(기준선)** |

### ARIMA의 수학적 모델

```
ARIMA(p, d, q):
  AR(p) — 과거 p개 시점의 값으로 예측 (AutoRegressive)
  I(d)  — d번 차분하여 비정상 시계열 → 정상 시계열 변환 (Integrated)
  MA(q) — 과거 q개 시점의 오차로 보정 (Moving Average)

  수식: y'(t) = c + φ₁y'(t-1) + φ₂y'(t-2) + θ₁ε(t-1) + ε(t)
        (여기서 y' = 1차 차분된 시계열)
```

> **참고**: ARIMA는 AI/딥러닝이 아닌 **순수 통계 모델**입니다. 다른 모델(Prophet, LSTM, TFT 등)의 성능을 평가하는 **Baseline**으로 사용합니다.

---
## Step 0. 라이브러리 설치 및 Import

In [ ]:
!pip install -q statsmodels scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error

print('All libraries loaded successfully!')

---
## Step 1. Shredder Sensor Data Generation (Daily Average)

산업용 슈레더(Shredder)의 센서 데이터를 시뮬레이션한 후, **일 단위 평균**으로 집계합니다.

### 왜 일 단위로 집계하는가?

ARIMA는 **저빈도 데이터**에서 가장 잘 작동합니다:
- 시간별 데이터(2,160개)는 계절성이 복잡하여 ARIMA에 비효율적
- 일별 평균(90개)으로 변환하면 트렌드와 주간 패턴이 명확해짐
- 고빈도 데이터는 LSTM, TFT 같은 딥러닝 모델이 더 적합

### 생성되는 센서 데이터

| 센서 | 단위 | 정상 범위 | 설명 |
|------|------|-----------|------|
| temperature | C | 20~38 | 베어링 온도 (예측 대상) |
| vibration | mm/s | 1.5~4.0 | 진동 RMS |
| current | A | 65~110 | 모터 전류 |
| rpm | rpm | 19~21 | 회전 속도 |
| throughput | t/h | 1.5~3.5 | 처리량 |

### 데이터에 포함된 패턴

- **일간 패턴**: 낮에 가동 -> 온도 상승, 밤에 정지 -> 온도 하강
- **주간 패턴**: 주말 비가동 -> 온도/전류 하강
- **마모 트렌드**: 칼날 마모로 90일간 온도가 서서히 상승 (+2.7C)
- **이상 이벤트**: 0.5% 확률로 고온 스파이크 발생 (+15~30C)

In [ ]:
def generate_shredder_data(days=90, freq_minutes=10, seed=42):
    """
    Shredder sensor data simulator.
    Generates realistic bearing temperature, vibration, current, rpm, throughput.
    """
    np.random.seed(seed)

    n_points = days * 24 * 60 // freq_minutes
    timestamps = pd.date_range(
        start='2026-01-01',
        periods=n_points,
        freq=f'{freq_minutes}min'
    )

    t = np.arange(n_points)
    hours = np.array([ts.hour for ts in timestamps])
    dow = np.array([ts.dayofweek for ts in timestamps])

    # === Bearing Temperature ===
    base_temp = 28.0
    daily_pattern = 5.0 * np.sin(2 * np.pi * hours / 24 - np.pi/2)
    weekly_pattern = np.where(dow >= 5, -3.0, 0.0)
    wear_trend = 0.03 * t / (24 * 60 / freq_minutes)
    noise = np.random.normal(0, 0.8, n_points)

    anomaly_mask = np.random.random(n_points) < 0.005
    anomaly_spike = anomaly_mask * np.random.uniform(15, 30, n_points)

    temperature = base_temp + daily_pattern + weekly_pattern + wear_trend + noise + anomaly_spike
    temperature = np.clip(temperature, 15, 80)

    # === Vibration RMS ===
    base_vib = 2.5
    vib_daily = 0.5 * np.sin(2 * np.pi * hours / 24)
    vib_wear = 0.02 * t / (24 * 60 / freq_minutes)
    vib_noise = np.random.normal(0, 0.3, n_points)
    vib_anomaly = anomaly_mask * np.random.uniform(5, 15, n_points)

    vibration = base_vib + vib_daily + vib_wear + vib_noise + vib_anomaly
    vibration = np.clip(vibration, 0.5, 25)

    # === Motor Current ===
    base_cur = 85.0
    cur_daily = 10.0 * np.sin(2 * np.pi * hours / 24 - np.pi/3)
    cur_wear = 0.05 * t / (24 * 60 / freq_minutes)
    cur_noise = np.random.normal(0, 2.0, n_points)
    cur_weekend = np.where(dow >= 5, -30.0, 0.0)

    current = base_cur + cur_daily + cur_wear + cur_noise + cur_weekend
    current = np.clip(current, 20, 150)

    # === RPM ===
    base_rpm = 20.0
    rpm_var = np.random.normal(0, 0.3, n_points)
    rpm_weekend = np.where(dow >= 5, -15.0, 0.0)

    rpm = base_rpm + rpm_var + rpm_weekend
    rpm = np.clip(rpm, 0, 25)

    # === Throughput ===
    base_tp = 2.5
    tp_daily = 0.5 * np.sin(2 * np.pi * hours / 24 - np.pi/4)
    tp_noise = np.random.normal(0, 0.15, n_points)
    tp_weekend = np.where(dow >= 5, -2.0, 0.0)

    throughput = base_tp + tp_daily + tp_noise + tp_weekend
    throughput = np.clip(throughput, 0, 4)

    df = pd.DataFrame({
        'timestamp': timestamps,
        'temperature': np.round(temperature, 2),
        'vibration': np.round(vibration, 2),
        'current': np.round(current, 2),
        'rpm': np.round(rpm, 2),
        'throughput': np.round(throughput, 2)
    })

    return df

print('generate_shredder_data() defined.')

In [ ]:
# Generate 90 days of hourly data, then resample to daily mean
df_hourly = generate_shredder_data(days=90, freq_minutes=60)

# Resample to daily average (ARIMA works best with lower frequency)
daily = df_hourly.set_index('timestamp').resample('D')['temperature'].mean().reset_index()
daily.columns = ['date', 'temperature']

print(f'Hourly data: {len(df_hourly)} samples')
print(f'Daily average: {len(daily)} days')
print(f'Period: {daily["date"].min().date()} ~ {daily["date"].max().date()}')
print(f'\nDaily temperature statistics:')
daily['temperature'].describe().round(2)

---
## Step 2. Raw Data Visualization

일 단위로 집계된 온도 데이터를 확인합니다.

**관찰 포인트:**
- 90일간 온도가 서서히 상승하는 **마모 트렌드**가 보이는가?
- 주말마다 온도가 하락하는 **주간 패턴**이 보이는가?
- 일 평균으로 집계했기 때문에 이상 스파이크가 완화됨

In [ ]:
# Daily temperature overview
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily['date'], daily['temperature'], 'b-o', markersize=4, linewidth=1.5, alpha=0.8)

# Highlight trend with linear fit
z = np.polyfit(np.arange(len(daily)), daily['temperature'], 1)
trend_line = np.poly1d(z)(np.arange(len(daily)))
ax.plot(daily['date'], trend_line, 'r--', linewidth=2, alpha=0.7,
        label=f'Trend: +{z[0]:.3f} C/day')

ax.set_title('Daily Average Bearing Temperature — 90 Days Overview', fontsize=14, fontweight='bold')
ax.set_ylabel('Temperature (C)')
ax.set_xlabel('Date')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Temperature trend: +{z[0]:.4f} C/day ({z[0]*90:.2f} C over 90 days)')

### 처음 14일 확대

처음 14일을 확대하여 주간 패턴(주말 하락)을 명확히 확인합니다.

In [ ]:
# Zoom in: first 14 days
first_14 = daily.iloc[:14]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(first_14['date'], first_14['temperature'], 'b-o', markersize=6, linewidth=2)

# Mark weekends
for i, row in first_14.iterrows():
    if row['date'].dayofweek >= 5:
        ax.axvspan(row['date'] - pd.Timedelta(hours=12),
                   row['date'] + pd.Timedelta(hours=12),
                   alpha=0.15, color='gray')

ax.set_title('Daily Temperature — First 14 Days (Weekend Dips Visible)', fontsize=13, fontweight='bold')
ax.set_ylabel('Temperature (C)')
ax.set_xlabel('Date')
ax.grid(True, alpha=0.3)

# Add weekend label
ax.annotate('Weekend', xy=(pd.Timestamp('2026-01-03'), first_14['temperature'].min()),
            fontsize=10, color='gray', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 3. ARIMA Concept Explanation

### ARIMA(p, d, q)란?

ARIMA는 3가지 구성요소의 조합입니다:

| 파라미터 | 이름 | 의미 | 이 시뮬레이션 |
|:---:|:---:|:---:|:---:|
| **p = 2** | Auto-Regressive | 과거 **2일**의 온도 값을 참조하여 예측 | "어제와 그저께 온도가 높았으면 오늘도 높을 것" |
| **d = 1** | Integrated (Differencing) | **1차 차분**으로 트렌드를 제거 | "온도 자체가 아닌 **변화량**을 예측" |
| **q = 1** | Moving Average | 과거 **1일**의 예측 오차를 보정에 활용 | "어제 예측이 틀린 만큼 오늘 보정" |

### 수학적 표현

$$y'_t = c + \phi_1 y'_{t-1} + \phi_2 y'_{t-2} + \theta_1 \varepsilon_{t-1} + \varepsilon_t$$

여기서:
- $y'_t = y_t - y_{t-1}$ (1차 차분)
- $\phi_1, \phi_2$ = AR 계수 (과거 값의 가중치)
- $\theta_1$ = MA 계수 (과거 오차의 가중치)
- $\varepsilon_t$ = 백색 잡음 (예측 불가능한 랜덤 요소)

### ARIMA가 "Baseline"인 이유

ARIMA는 **가장 단순한 통계 모델**이므로:
- ARIMA보다 나쁜 AI 모델 = 도입할 가치 없음
- ARIMA보다 좋은 AI 모델 = "ARIMA 대비 X C 개선" 으로 효과 입증

---
## Step 4. Stationarity Check & Differencing

ARIMA의 핵심 전제: 시계열이 **정상(Stationary)**이어야 합니다.

### 정상 시계열이란?
- **평균**이 시간에 따라 변하지 않음
- **분산**이 시간에 따라 변하지 않음
- 우리 데이터에는 마모 트렌드(온도 상승)가 있으므로 **비정상**

### 해결: 차분(Differencing, d=1)
- 원본: $y_t$ (트렌드 있음 = 비정상)
- 1차 차분: $y'_t = y_t - y_{t-1}$ (트렌드 제거 = 정상)

In [ ]:
# Original series vs 1st difference
diff = daily['temperature'].diff().dropna()

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# Original series (non-stationary: has trend)
axes[0].plot(daily['date'], daily['temperature'], 'b-o', markersize=3, linewidth=1)
axes[0].set_title('Original Series — Non-Stationary (has upward trend)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Temperature (C)')
axes[0].grid(True, alpha=0.3)

# 1st difference (stationary: trend removed)
axes[1].plot(daily['date'][1:], diff.values, 'purple', alpha=0.7, linewidth=1)
axes[1].axhline(y=0, color='black', linewidth=1)
axes[1].axhline(y=diff.mean(), color='red', linestyle='--', linewidth=1.5,
                label=f'Mean: {diff.mean():.3f} C/day')
axes[1].set_title('1st Difference (d=1) — Stationary (trend removed)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Daily Change (C)')
axes[1].set_xlabel('Date')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Original series mean: {daily["temperature"].mean():.2f} C (varies over time = non-stationary)')
print(f'1st difference mean:  {diff.mean():.4f} C/day (near zero = stationary)')
print(f'1st difference std:   {diff.std():.4f} C')

### 자기상관 함수 (ACF)

자기상관(Autocorrelation)은 "과거 며칠 전 데이터가 현재와 얼마나 관련 있는가?"를 측정합니다.

- **Lag 1**: 어제 온도와 오늘 온도의 상관관계
- **Lag 7**: 일주일 전 온도와 오늘 온도의 상관관계 (주간 패턴)
- 빨간 점선 밖의 값만 통계적으로 유의미

In [ ]:
# Autocorrelation Function (ACF)
lags = range(1, 22)
acf_vals = [daily['temperature'].autocorr(lag=l) for l in lags]

fig, ax = plt.subplots(figsize=(12, 4))
colors = ['steelblue' if l != 7 and l != 14 else 'red' for l in lags]
ax.bar(lags, acf_vals, color=colors, alpha=0.7, edgecolor='white')
ax.axhline(y=0, color='black', linewidth=0.5)
ax.axhline(y=1.96/np.sqrt(len(daily)), color='red', linestyle='--', alpha=0.5,
           label='95% Confidence Interval')
ax.axhline(y=-1.96/np.sqrt(len(daily)), color='red', linestyle='--', alpha=0.5)

ax.set_title('Autocorrelation Function (ACF) — "How does past temperature relate to today?"',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Lag (days)')
ax.set_ylabel('Autocorrelation')
ax.set_xticks(list(lags))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print('Key observations:')
print(f'  Lag 1 (yesterday):     {acf_vals[0]:.3f}')
print(f'  Lag 7 (1 week ago):    {acf_vals[6]:.3f}')
print(f'  Lag 14 (2 weeks ago):  {acf_vals[13]:.3f}')

---
## Step 5. Train/Test Split + Model Training

시계열 데이터는 **반드시 시간순으로 분할**해야 합니다. 랜덤 분할은 미래 정보가 학습에 포함되어 성능이 과대 추정됩니다.

```
|<--- Train (75%) --->|<--- Test (25%) --->|
|   Jan ~ early Mar    |  mid Mar ~ end Mar  |
```

### ARIMA(2, 1, 1) 파라미터 선택 근거
- **p=2**: ACF에서 Lag 1~2가 유의미 -> 과거 2일 참조
- **d=1**: 마모 트렌드 제거를 위한 1차 차분
- **q=1**: 직전 예측 오차 1개로 보정

In [ ]:
# Time-ordered split (75% train / 25% test)
train_size = int(len(daily) * 0.75)
train = daily.iloc[:train_size]
test = daily.iloc[train_size:]

print(f'Train: {len(train)} days ({train["date"].min().date()} ~ {train["date"].max().date()})')
print(f'Test : {len(test)} days ({test["date"].min().date()} ~ {test["date"].max().date()})')

In [ ]:
# ARIMA(2,1,1) model training
print('ARIMA(2,1,1) model fitting...')
print('  p=2: AutoRegressive — use past 2 days')
print('  d=1: Differencing — remove trend')
print('  q=1: Moving Average — use past 1 error')

model = ARIMA(train['temperature'].values, order=(2, 1, 1))
fitted = model.fit()

print(f'\nModel trained successfully!')
print(f'  AIC: {fitted.aic:.2f} (lower = better model)')
print(f'  BIC: {fitted.bic:.2f}')
print(f'\nModel coefficients:')
print(f'  AR(1) = {fitted.arparams[0]:.4f}')
print(f'  AR(2) = {fitted.arparams[1]:.4f}')
print(f'  MA(1) = {fitted.maparams[0]:.4f}')

In [ ]:
# Forecast test period
forecast = fitted.forecast(steps=len(test))

print(f'Forecast generated: {len(forecast)} days')
print(f'Forecast range: {forecast.min():.2f} ~ {forecast.max():.2f} C')

---
## Step 6. Performance Evaluation

테스트 구간의 실제값과 예측값을 비교합니다.

| 지표 | 의미 |
|------|------|
| **MAE** | 평균 절대 오차 — 평균적으로 몇 도 틀리는가 |
| **RMSE** | 평균 제곱근 오차 — 큰 오차에 더 민감 |

In [ ]:
mae = mean_absolute_error(test['temperature'], forecast)
rmse = np.sqrt(np.mean((test['temperature'].values - forecast.values)**2))

print('=' * 50)
print('  ARIMA(2,1,1) Performance Evaluation')
print('=' * 50)
print(f'  MAE  = {mae:.2f} C (average error)')
print(f'  RMSE = {rmse:.2f} C')
print('=' * 50)
print(f'\n  Baseline established!')
print(f'  Other models should beat MAE = {mae:.2f} C to justify their complexity.')

---
## Step 7. Result Visualization

### 7-1. Full Forecast vs Actual

전체 기간의 실제 온도와 테스트 구간의 예측값을 비교합니다.
분홍색 영역은 **신뢰 구간(+/-2 sigma)**으로, 실제값이 이 안에 들어올 확률이 약 95%입니다.

In [ ]:
# 7-1. Full forecast vs actual with confidence interval
residual_std = (test['temperature'].values - forecast.values).std()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train['date'], train['temperature'], 'b-o', markersize=3, linewidth=1, alpha=0.7, label='Train (Actual)')
ax.plot(test['date'], test['temperature'], 'g-o', markersize=3, linewidth=1, alpha=0.7, label='Test (Actual)')
ax.plot(test['date'], forecast.values, 'r--', linewidth=2, label='ARIMA Forecast')
ax.fill_between(test['date'],
                forecast.values - 2*residual_std,
                forecast.values + 2*residual_std,
                alpha=0.15, color='red',
                label=f'Confidence Interval (+/- 2 sigma = {2*residual_std:.1f} C)')
ax.axvline(x=train['date'].iloc[-1], color='gray', linestyle=':', linewidth=1.5, label='Train/Test boundary')

ax.set_title(f'ARIMA(2,1,1) Forecast — Daily Average Bearing Temperature (MAE={mae:.2f} C)',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Temperature (C)')
ax.set_xlabel('Date')
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7-2. ACF Bar Chart

자기상관 함수(ACF)를 막대 차트로 시각화합니다. 각 Lag에서의 상관관계를 확인하세요.
빨간 막대(Lag 7, 14)는 주간 패턴을 나타냅니다.

In [ ]:
# 7-2. ACF bar chart
lags_plot = range(1, 22)
acf_values = [daily['temperature'].autocorr(lag=l) for l in lags_plot]

fig, ax = plt.subplots(figsize=(12, 4))
bar_colors = ['#e74c3c' if l % 7 == 0 else 'steelblue' for l in lags_plot]
bars = ax.bar(lags_plot, acf_values, color=bar_colors, alpha=0.7, edgecolor='white')
ax.axhline(y=0, color='black', linewidth=0.5)
ax.axhline(y=1.96/np.sqrt(len(daily)), color='red', linestyle='--', alpha=0.5,
           label='95% Confidence Interval')
ax.axhline(y=-1.96/np.sqrt(len(daily)), color='red', linestyle='--', alpha=0.5)

ax.set_title('ACF — Autocorrelation at Each Lag (Red = Weekly Pattern at Lag 7, 14)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Lag (days)')
ax.set_ylabel('Autocorrelation Coefficient')
ax.set_xticks(list(lags_plot))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

### 7-3. 1st Difference Plot

1차 차분(d=1)으로 트렌드가 제거된 결과를 시각화합니다.
차분 후 평균이 0 근처에 머물면 정상 시계열로 변환된 것입니다.

In [ ]:
# 7-3. 1st Difference plot
diff_train = train['temperature'].diff().dropna()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(train['date'][1:], diff_train.values, 'purple', alpha=0.7, linewidth=1)
ax.axhline(y=0, color='black', linewidth=1)
ax.axhline(y=diff_train.mean(), color='red', linestyle='--', linewidth=1.5,
           label=f'Mean: {diff_train.mean():.3f} C/day')
ax.axhline(y=2*diff_train.std(), color='orange', linestyle=':', alpha=0.5,
           label=f'+/- 2 std ({2*diff_train.std():.2f} C)')
ax.axhline(y=-2*diff_train.std(), color='orange', linestyle=':', alpha=0.5)

ax.set_title('1st Difference (d=1) — Trend Removal Result', fontsize=13, fontweight='bold')
ax.set_ylabel('Daily Temperature Change (C)')
ax.set_xlabel('Date')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'After differencing:')
print(f'  Mean: {diff_train.mean():.4f} C/day (near zero = trend removed)')
print(f'  Std:  {diff_train.std():.4f} C')

### 7-4. Error Distribution

예측 오차(실제 - 예측)의 분포를 히스토그램으로 확인합니다.
정규분포에 가까우면 모델이 체계적 편향 없이 잘 작동하고 있다는 의미입니다.

In [ ]:
# 7-4. Error distribution histogram
errors = test['temperature'].values - forecast.values

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(errors, bins=15, color='steelblue', alpha=0.7, edgecolor='white', density=True)
ax.axvline(x=0, color='red', linewidth=2, linestyle='--', label='Zero error')
ax.axvline(x=np.mean(errors), color='orange', linewidth=2,
           label=f'Mean error: {np.mean(errors):.2f} C')

ax.set_title('Forecast Error Distribution (Actual - Predicted)', fontsize=13, fontweight='bold')
ax.set_xlabel('Error (C)')
ax.set_ylabel('Density')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Error statistics:')
print(f'  Mean  : {np.mean(errors):+.2f} C')
print(f'  Std   : {np.std(errors):.2f} C')
print(f'  Median: {np.median(errors):+.2f} C')
print(f'  Within +/- 2 C: {(np.abs(errors) < 2).mean()*100:.1f}%')
print(f'  Within +/- 3 C: {(np.abs(errors) < 3).mean()*100:.1f}%')

### 7-5. Model Comparison Table

ARIMA를 Baseline으로 놓고, 다른 AI 모델이 얼마나 개선하는지 비교합니다.
(Prophet, LSTM 등의 MAE 값은 해당 노트북에서 확인 가능)

In [ ]:
# 7-5. Model comparison table
# ARIMA as baseline, other models for reference
models = {
    'ARIMA(2,1,1)': {'mae': mae, 'type': 'Statistics', 'variables': 'Univariate', 'role': 'BASELINE'},
    'Prophet-Style': {'mae': 1.5, 'type': 'Statistics', 'variables': 'Univariate', 'role': 'Trend/Seasonal'},
    'LSTM Autoencoder': {'mae': None, 'type': 'Deep Learning', 'variables': 'Multivariate', 'role': 'Anomaly Detection'},
    'XGBoost': {'mae': 0.8, 'type': 'Machine Learning', 'variables': 'Multivariate', 'role': 'Feature-based'},
    'TFT': {'mae': 0.5, 'type': 'Deep Learning', 'variables': 'Multivariate', 'role': 'Best Accuracy'},
}

fig, ax = plt.subplots(figsize=(10, 5))

model_names = []
mae_values = []
bar_colors = []
for name, info in models.items():
    if info['mae'] is not None:
        model_names.append(name)
        mae_values.append(info['mae'])
        if name == 'ARIMA(2,1,1)':
            bar_colors.append('#e74c3c')  # red for baseline
        else:
            bar_colors.append('#3498db')

bars = ax.barh(model_names, mae_values, color=bar_colors, alpha=0.8, edgecolor='white')
ax.axvline(x=mae, color='red', linestyle='--', alpha=0.5, label=f'ARIMA Baseline ({mae:.2f} C)')

# Add value labels
for bar, val in zip(bars, mae_values):
    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2,
            f'{val:.2f} C', va='center', fontsize=11, fontweight='bold')

ax.set_title('Model Comparison — MAE (Lower = Better)', fontsize=13, fontweight='bold')
ax.set_xlabel('MAE (C)')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='x')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('모델 비교 (ARIMA = Baseline):')
print(f'{"Model":<20} {"Type":<15} {"Variables":<15} {"MAE":>8} {"vs ARIMA":>12}')
print('-' * 70)
for name, info in models.items():
    mae_str = f'{info["mae"]:.2f} C' if info["mae"] is not None else 'N/A'
    if info["mae"] is not None and name != 'ARIMA(2,1,1)':
        diff = mae - info["mae"]
        diff_str = f'-{diff:.2f} C' if diff > 0 else f'+{abs(diff):.2f} C'
    elif name == 'ARIMA(2,1,1)':
        diff_str = '(baseline)'
    else:
        diff_str = '(different task)'
    print(f'{name:<20} {info["type"]:<15} {info["variables"]:<15} {mae_str:>8} {diff_str:>12}')

### 7-6. Residual Analysis

잔차(Residual = 실제 - 예측)를 시간 순으로 시각화합니다.
패턴 없이 랜덤하게 분포하면 모델이 정보를 잘 추출한 것입니다.

In [ ]:
# 7-6. Residual analysis (residuals over time)
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(test['date'], errors, 'o-', color='purple', markersize=4, linewidth=1, alpha=0.7)
ax.axhline(y=0, color='black', linewidth=1)
ax.axhline(y=2*np.std(errors), color='red', linestyle='--', alpha=0.5,
           label=f'+/- 2 std ({2*np.std(errors):.2f} C)')
ax.axhline(y=-2*np.std(errors), color='red', linestyle='--', alpha=0.5)
ax.fill_between(test['date'], -2*np.std(errors), 2*np.std(errors),
                alpha=0.1, color='red')

ax.set_title('Residual Analysis — Forecast Errors Over Time', fontsize=13, fontweight='bold')
ax.set_ylabel('Residual (C)')
ax.set_xlabel('Date')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Check for patterns in residuals
print('Residual diagnostics:')
print(f'  Mean: {np.mean(errors):.4f} C (should be near 0)')
print(f'  Std:  {np.std(errors):.4f} C')
print(f'  Lag-1 autocorrelation: {pd.Series(errors).autocorr(lag=1):.3f} (should be near 0)')
print(f'  Points outside +/-2 std: {(np.abs(errors) > 2*np.std(errors)).sum()} / {len(errors)}')

---
## Step 8. Summary

### ARIMA(p, d, q) 파라미터 의미

| 파라미터 | 값 | 의미 | 슈레더 해석 |
|:---:|:---:|:---:|:---:|
| **p** (AR) | 2 | 과거 2일의 온도를 참조 | "어제/그저께 온도로 오늘 예측" |
| **d** (I) | 1 | 1차 차분으로 트렌드 제거 | "마모에 의한 온도 상승 트렌드 보정" |
| **q** (MA) | 1 | 과거 1일의 오차를 보정 | "어제 예측 오차만큼 오늘 보정" |

### ARIMA의 역할: Baseline 모델

ARIMA는 가장 단순한 통계 모델로, 다른 AI 모델의 **성능 기준선**입니다:
- AI 모델의 MAE < ARIMA의 MAE -> AI 도입 가치 있음
- AI 모델의 MAE >= ARIMA의 MAE -> AI 도입 근거 부족

### 장점
- **순수 통계** — AI/딥러닝 없이 수학적 원리로 예측
- **적은 데이터로 작동** — 수십~수백 건만으로도 학습 가능
- **수학적으로 명확** — p, d, q 파라미터의 의미가 투명
- **Baseline으로 최적** — 모든 시계열 프로젝트의 첫 번째 모델

### 단점
- **단변량만 가능** — 온도 하나만 사용 (진동, 전류 동시 활용 불가)
- **비선형 패턴 학습 불가** — 복잡한 센서 간 상호작용 포착 못함
- **장기 예측 정확도 급락** — 예측 기간이 길어질수록 성능 저하
- **고빈도 데이터에 비효율** — 초/분 단위 데이터는 집계 필요

### 슈레더 현장 적용
- **적합**: 다른 AI 모델의 Baseline(기준선)으로 사용
- **활용**: "AI 모델 X가 ARIMA 대비 MAE를 Y C 개선했습니다" -> AI 도입 근거
- **부적합**: 다변량 이상 탐지 (-> LSTM), 복잡한 패턴 예측 (-> TFT)

---

> **다음 단계**: `01_Prophet` 폴더에서 트렌드/계절성 분해 예측을, `02_LSTM` 폴더에서 다변량 이상 탐지를 체험해 보세요.

In [ ]:
print('=' * 60)
print('  ARIMA Simulation Complete!')
print('=' * 60)
print(f'''
  Model: ARIMA(2, 1, 1) — Statistical Baseline

  ARIMA(p, d, q) Parameters:
    p=2 (AR) : Use past 2 days' temperature
    d=1 (I)  : 1st differencing to remove trend
    q=1 (MA) : Use past 1 forecast error for correction

  Data:
    {len(daily)} daily averages ({daily["date"].min().date()} ~ {daily["date"].max().date()})
    Train: {len(train)} days / Test: {len(test)} days

  Model Quality:
    AIC: {fitted.aic:.2f}
    BIC: {fitted.bic:.2f}

  Performance (Baseline):
    MAE  = {mae:.2f} C
    RMSE = {rmse:.2f} C

  Baseline Role:
    AI models that beat MAE = {mae:.2f} C justify their added complexity.
    "AI model X improved MAE by Y C over ARIMA baseline"
''')